# Large Retail Data Set for EDA
**Dataset**: utkalk/large-retail-data-set-for-eda

> **Note**: 518MB file with 1M rows and 78 columns. Analysis uses sampling for performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load sample for analysis
df = pd.read_csv('abhinav/large-retail-eda/retail_data.csv', nrows=200000)
total_rows = sum(1 for _ in open('abhinav/large-retail-eda/retail_data.csv')) - 1
print(f'Total rows in file: {total_rows:,}')
print(f'Loaded sample: {len(df):,} rows x {df.shape[1]} cols')
print(f'\nColumns ({df.shape[1]}):')
for i, col in enumerate(df.columns):
    print(f'  {i+1}. {col} ({df[col].dtype})')


## 1. Data Quality

In [ ]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
quality = pd.DataFrame({'nulls': null_counts, 'null_%': null_pct, 'dtype': df.dtypes, 'unique': df.nunique()})
quality[quality['nulls'] > 0]


In [ ]:
# If no nulls
if df.isnull().sum().sum() == 0:
    print('No null values in the dataset — very clean!')
else:
    print(f'Total null values: {df.isnull().sum().sum():,}')


## 2. Numeric Summary

In [ ]:
df.describe()


## 3. Product Analysis

In [ ]:
prod_cols = [c for c in df.columns if any(k in c.lower() for k in ['product', 'item', 'sku', 'brand'])]
print('Product-related columns:', prod_cols)
for col in prod_cols:
    print(f'\n{col} — {df[col].nunique()} unique values')
    print(df[col].value_counts().head(10))


In [ ]:
# Product category distribution
cat_cols = [c for c in df.columns if 'category' in c.lower()]
if cat_cols:
    for col in cat_cols:
        fig, ax = plt.subplots(figsize=(12, 6))
        df[col].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('Set2'))
        ax.set_title(f'{col} Distribution')
        ax.set_ylabel('Count')
        plt.tight_layout()
        plt.show()


## 4. Sales & Revenue Analysis

In [ ]:
sales_cols = [c for c in df.columns if any(k in c.lower() for k in ['price', 'sales', 'revenue', 'cost', 'amount', 'total', 'profit', 'discount'])]
print('Sales-related columns:', sales_cols)

num_sales = [c for c in sales_cols if df[c].dtype in ['float64', 'int64']]
if num_sales:
    n = min(len(num_sales), 6)
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 4*((n+1)//2)))
    axes = axes.flatten()
    for i, col in enumerate(num_sales[:n]):
        axes[i].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
        axes[i].set_title(f'{col} (mean={df[col].mean():.2f})')
        axes[i].axvline(df[col].mean(), color='red', linestyle='--')
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 5. Customer Analysis

In [ ]:
cust_cols = [c for c in df.columns if any(k in c.lower() for k in ['customer', 'user', 'client', 'loyalty', 'churn', 'age', 'gender', 'income'])]
print('Customer-related columns:', cust_cols)

# Demographics
demo_cols = [c for c in cust_cols if df[c].nunique() < 20]
if demo_cols:
    n = min(len(demo_cols), 6)
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 4*((n+1)//2)))
    axes = axes.flatten()
    for i, col in enumerate(demo_cols[:n]):
        df[col].value_counts().plot(kind='bar', ax=axes[i], color=sns.color_palette('pastel'))
        axes[i].set_title(col)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 6. Temporal Analysis

In [ ]:
date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
print('Date columns:', date_cols)

for col in date_cols[:2]:  # first 2 date cols
    try:
        dt = pd.to_datetime(df[col], errors='coerce')
        if dt.notna().sum() > 0:
            print(f'\n{col}: {dt.min()} to {dt.max()}')
            fig, axes = plt.subplots(1, 2, figsize=(16, 5))
            dt.dt.dayofweek.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
            axes[0].set_title(f'{col} — Day of Week')
            dt.dt.month.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
            axes[1].set_title(f'{col} — Month')
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f'{col}: could not parse — {e}')


## 7. Shelf-Relevant Features

In [ ]:
shelf_cols = [c for c in df.columns if any(k in c.lower() for k in ['shelf', 'stock', 'store', 'location', 'promo', 'rating'])]
print('Shelf/store-relevant columns:', shelf_cols)

for col in shelf_cols:
    if df[col].nunique() <= 20:
        print(f'\n{col}:')
        print(df[col].value_counts())
    else:
        print(f'\n{col}: {df[col].nunique()} unique — Mean: {df[col].mean():.2f}' if df[col].dtype in ['float64','int64'] else f'\n{col}: {df[col].nunique()} unique')


## 8. Correlation Matrix (Key Numeric Columns)

In [ ]:
key_num = [c for c in df.select_dtypes(include=[np.number]).columns if any(k in c.lower() for k in ['price','sales','quantity','revenue','cost','discount','profit','rating','stock','age','income','loyalty'])]
if len(key_num) > 2:
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(df[key_num].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0)
    ax.set_title('Correlation Matrix — Key Business Metrics')
    plt.tight_layout()
    plt.show()


## 9. Relevance to Shelf Optimization / Planogram AI

**Strengths (78 columns!):**
- Product shelf life, stock levels, ratings — direct shelf management features
- Promotion data (type, channel, effectiveness) — promo shelf placement
- Customer demographics & loyalty — localized planogram potential
- Sales, revenue, cost, discount — full margin analysis
- Store location data — multi-store optimization

**Limitations:**
- Analysis on 200k sample — full 1M may reveal more patterns
- Likely synthetic data (too clean, no nulls)
- No physical shelf layout or aisle mapping
